In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv("../data/cafe_june_raw.csv")

raw

,outlet,date_text,daypart,revenue_raw,tickets,items,staff_on_shift,manager,manager_email,notes
0,RAFFLES PLACE,01/06/2025,evening,S$78.38,9.0,13.0,1.0,Aisha Rahman,aisha.rahman@dailygrind.sg,NaN
1,Raffles Place,01/06/2025,Midday,205.21,20.0,30.0,2.0,Aisha Rahman,aisha.rahman@dailygrind.sg,NaN
2,raffles place,01/06/2025,AM,324.52,50.0,71.0,3.0,Aisha Rahman,aisha.rahman@dailygrind.sg,N.A.
3,Tampines Mall,01/06/2025,PM,615.79,61.0,90.0,4.0,wei ming tan,weiming.tan@dailygrind.sg,NaN
4,Tampines Mall,01/06/2025,Midday,932.35,82.0,119.0,5.0,wei ming tan,weiming.tan@dailygrind.sg,NaN
...,...,...,...,...,...,...,...,...,...,...
361,Marina Bay,30/06/2025,Midday,S$497.02,49.0,73.0,3.0,Priya Nair,priya.nair@dailygrind.sg,NaN
362,marina bay,30/06/2025,Morning,S$548.02,90.0,135.0,5.0,Priya Nair,priya.nair@dailygrind.sg,NaN
363,Holland V,30/06/2025,Evening,619.51,66.0,91.0,4.0,DANIEL LIM,daniel.lim@dailygrind.com.sg,NaN
364,Holland Village,30/06/2025,midday,515.76,50.0,75.0,3.0,DANIEL LIM,daniel.lim@dailygrind.com.sg,NaN


In [ ]:
raw.info()  # check types and holes (shape = how big; dtypes = anything stored wrong)

raw.describe()  # check numbers

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   outlet          366 non-null    object 
 1   date_text       366 non-null    object 
 2   daypart         366 non-null    object 
 3   revenue_raw     363 non-null    object 
 4   tickets         364 non-null    float64
 5   items           363 non-null    float64
 6   staff_on_shift  361 non-null    float64
 7   manager         366 non-null    object 
 8   manager_email   364 non-null    object 
 9   notes           34 non-null     object 
dtypes: float64(3), object(7)
memory usage: 28.7+ KB


,tickets,items,staff_on_shift
count,364.000000,363.000000,361.000000
mean,57.912088,84.884298,3.479224
std,36.501601,53.516743,1.262817
min,-4.000000,10.000000,1.000000
25%,38.750000,54.500000,3.000000
50%,53.000000,76.000000,3.000000
75%,68.250000,101.000000,4.000000
max,216.000000,336.000000,6.000000


In [ ]:
raw.head() # check what do rows look like

,outlet,date_text,daypart,revenue_raw,tickets,items,staff_on_shift,manager,manager_email,notes
0,RAFFLES PLACE,01/06/2025,evening,S$78.38,9.0,13.0,1.0,Aisha Rahman,aisha.rahman@dailygrind.sg,NaN
1,Raffles Place,01/06/2025,Midday,205.21,20.0,30.0,2.0,Aisha Rahman,aisha.rahman@dailygrind.sg,NaN
2,raffles place,01/06/2025,AM,324.52,50.0,71.0,3.0,Aisha Rahman,aisha.rahman@dailygrind.sg,N.A.
3,Tampines Mall,01/06/2025,PM,615.79,61.0,90.0,4.0,wei ming tan,weiming.tan@dailygrind.sg,NaN
4,Tampines Mall,01/06/2025,Midday,932.35,82.0,119.0,5.0,wei ming tan,weiming.tan@dailygrind.sg,NaN


In [13]:
# 👉 `.copy()` makes an independent table, so `raw` stays as the untouched original --
#    which is what lets us compare "before and after" at the end of the session.
clean = raw.copy()

# 👉 Strip out everything that is not a digit, a dot or a minus sign: "S$1,240.50" -> "1240.50".
#    `regex=True` says "the thing I am searching for is a pattern, not literal text".
#    `to_numeric` then turns that text into a real number; `errors="coerce"` puts NaN
#    wherever the text could not be read as a number at all.
clean["revenue_sgd"] = pd.to_numeric(
    clean["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
)

clean[["revenue_raw", "revenue_sgd"]].head()

,revenue_raw,revenue_sgd
0,S$78.38,78.38
1,205.21,205.21
2,324.52,324.52
3,615.79,615.79
4,932.35,932.35


In [ ]:
import pandas as pd
import numpy as np

raw = pd.read_csv("../data/cafe_june_raw.csv")


# 👉 Recap of Part 2: rebuild `clean` from `raw` so this notebook runs on its own.
#    (If you already ran Part_2_data_quality.ipynb in this kernel, this just repeats the same result.)
clean = raw.copy()

# remove removes $, SGD, commas, spaces etc and keep numbers, na -> NaN
clean["revenue_sgd"] = pd.to_numeric(
    clean["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
)

# standardize notes from - and N.A. to NaN
clean["notes"] = clean["notes"].replace(["N.A.", "-"], np.nan)

# fill missing staff on shift and items value with median
clean = clean.fillna({
    "staff_on_shift": clean["staff_on_shift"].median(),
    "items": clean["items"].median(),
})

# remove duplicate rows
clean = clean.drop_duplicates()

# set invalid revenue < 0 & > 20,000 to NaN (default)
clean["revenue_sgd"] = clean["revenue_sgd"].mask(
    (clean["revenue_sgd"] < 0) | (clean["revenue_sgd"] > 20000)
)

# set invalid tickets value <= 0 to NaN (default)
clean["tickets"] = clean["tickets"].mask(clean["tickets"] <= 0)

# fill NaN revenue and tickets value with median
clean = clean.fillna({
    "revenue_sgd": clean["revenue_sgd"].median(),
    "tickets": clean["tickets"].median(),
})

clean

,outlet,date_text,daypart,revenue_raw,tickets,items,staff_on_shift,manager,manager_email,notes,revenue_sgd
0,RAFFLES PLACE,01/06/2025,evening,S$78.38,9.0,13.0,1.0,Aisha Rahman,aisha.rahman@dailygrind.sg,NaN,78.38
1,Raffles Place,01/06/2025,Midday,205.21,20.0,30.0,2.0,Aisha Rahman,aisha.rahman@dailygrind.sg,NaN,205.21
2,raffles place,01/06/2025,AM,324.52,50.0,71.0,3.0,Aisha Rahman,aisha.rahman@dailygrind.sg,NaN,324.52
3,Tampines Mall,01/06/2025,PM,615.79,61.0,90.0,4.0,wei ming tan,weiming.tan@dailygrind.sg,NaN,615.79
4,Tampines Mall,01/06/2025,Midday,932.35,82.0,119.0,5.0,wei ming tan,weiming.tan@dailygrind.sg,NaN,932.35
...,...,...,...,...,...,...,...,...,...,...,...
361,Marina Bay,30/06/2025,Midday,S$497.02,49.0,73.0,3.0,Priya Nair,priya.nair@dailygrind.sg,NaN,497.02
362,marina bay,30/06/2025,Morning,S$548.02,90.0,135.0,5.0,Priya Nair,priya.nair@dailygrind.sg,NaN,548.02
363,Holland V,30/06/2025,Evening,619.51,66.0,91.0,4.0,DANIEL LIM,daniel.lim@dailygrind.com.sg,NaN,619.51
364,Holland Village,30/06/2025,midday,515.76,50.0,75.0,3.0,DANIEL LIM,daniel.lim@dailygrind.com.sg,NaN,515.76


In [31]:
clean.loc[212,:]

outlet            Tampines  Mall
date_text             18/06/2025
daypart                  evening
revenue_raw             S$376.49
tickets                     40.0
items                       61.0
staff_on_shift               3.0
manager             wei ming tan
manager_email                NaN
notes                        NaN
revenue_sgd               376.49
Name: 212, dtype: object

In [18]:
display(raw["items"].median())

display(raw["tickets"].median())

display(raw["staff_on_shift"].median())

76.0

53.0

3.0

### Handling Duplicates Examples

In [20]:
import pandas as pd

# 1. Create the data representing the kiosk glitch
clinic = {
    'Patiend_ID': ['P-987', 'P-987', 'P-987','P-988', 'P-988'],           
    'Reason': ['Flu', 'Flu', 'Cold', 'Cough', 'Fever'], 
    'Qnumber': [14, 15, 16, 17, 18]                           
}

# 2. Convert it into a Pandas DataFrame
qticket = pd.DataFrame(clinic)

print("--- Original Data (Clinic Q number) ---")
display(qticket) # Using display() for nice formatting in Colab

# 3. Apply the drop_duplicates function
qticket_clean = qticket.drop_duplicates(subset=["Patiend_ID"])

print("\n--- Cleaned Q number ---")
display(qticket_clean)

--- Original Data (Clinic Q number) ---


,Patiend_ID,Reason,Qnumber
0,P-987,Flu,14
1,P-987,Flu,15
2,P-987,Cold,16
3,P-988,Cough,17
4,P-988,Fever,18



--- Cleaned Q number ---


,Patiend_ID,Reason,Qnumber
0,P-987,Flu,14
3,P-988,Cough,17


In [21]:
# keep last Q number
qticket_clean = qticket.drop_duplicates(subset=["Patiend_ID"], keep="last")
display(qticket_clean)

,Patiend_ID,Reason,Qnumber
2,P-987,Cold,16
4,P-988,Fever,18


In [24]:
import pandas as pd

# 1. Recreate the original dataset from your screenshot
result = {
    'Student_ID': ['S1', 'S2', 'S1', 'S2', 'S1', 'S2', 'S2', 'S2', 'S1'],
    'Quiz_ID': [1, 1, 2, 3, 3, 4, 4, 4, 1],
    'Score': [80, 92, 52, 63, 84, 75, 86, 66, 98]
}

quiz_result = pd.DataFrame(result)

print("--- Original Data ---")
display(quiz_result)

# 2. Drop duplicates based on same student and quiz, keeping the LAST entry
cleaned_quiz_result = quiz_result.drop_duplicates(subset=["Student_ID", "Quiz_ID"], keep="last")

print("\n--- Cleaned Data (Last submission kept) ---")
display(cleaned_quiz_result)


--- Original Data ---


,Student_ID,Quiz_ID,Score
0,S1,1,80
1,S2,1,92
2,S1,2,52
3,S2,3,63
4,S1,3,84
5,S2,4,75
6,S2,4,86
7,S2,4,66
8,S1,1,98



--- Cleaned Data (Last submission kept) ---


,Student_ID,Quiz_ID,Score
1,S2,1,92
2,S1,2,52
3,S2,3,63
4,S1,3,84
7,S2,4,66
8,S1,1,98


In [30]:
# keep the highest score as final result, sort by Student and Quiz ID
highest_result = quiz_result.groupby(['Student_ID', 'Quiz_ID'])['Score'].idxmax()

# Filter the original dataframe using those index numbers
final_quiz_result = quiz_result.loc[highest_result]

print("\n--- Cleaned Data (highest score kept) ---")
display(final_quiz_result)


--- Cleaned Data (highest score kept) ---


,Student_ID,Quiz_ID,Score
8,S1,1,98
2,S1,2,52
4,S1,3,84
1,S2,1,92
3,S2,3,63
6,S2,4,86


In [6]:
# keep the highest score as final result
# 1. Sort by score in DESCENDING order (highest values at the top)
# 2. Then drop duplicates keeping the 'first' (which is now the highest!)
highest_sortquiz_result = quiz_result.sort_values(by='Score', ascending=False).drop_duplicates(subset=["Student_ID", "Quiz_ID"], keep="first")

display(highest_sortquiz_result)

,Student_ID,Quiz_ID,Score
8,S1,1,98
1,S2,1,92
6,S2,4,86
4,S1,3,84
3,S2,3,63
2,S1,2,52


In [ ]:
## Product Allocation Sample

import pandas as pd
import numpy as np

# 1. Recreate the dataset
data = [
    (1, 169535646, '21/6/2019', 532, 'Truck'),
    (1, 734265621, '18/9/2030', 5733, 'Van'),
    (1, 747557690, '19/8/2027', 677, 'Car'),
    (1, 847557635, '18/6/2024', 234, 'NA'),
    (3, 697949195, '19/4/2021', 623, 'Van'),
    (3, 797969175, '19/4/2022', 907, 'Van'),
    (3, 797670195, '17/7/2022', 2314, 'NA'),
    (3, 872231801, '19/7/2026', 566, 'Car'),
    (4, 971694618, '19/1/2016', 788, 'Truck'),
    (4, 991694620, '1/7/2020', 4543, 'Van'),
    (10, 971696623, '28/5/2020', 432, 'NA'),
    (2, 166535746, '12/3/2027', 332, 'NA'),
    (2, 235265625, '21/1/2018', 733, 'NA'),
    (6, 275330646, '12/3/2027', 1332, 'NA'),
    (6, 635265643, '19/12/2022', 2475, 'NA'),
    (6, 215265656, '22/10/2012', 888, 'Van'),
    (11, 268535746, '21/7/2017', 5232, 'Truck'),
    (11, 334265625, '11/10/2028', 333, 'NA')
]

columns = ['client_id', 'Claim_id', 'claim_date', 'claim_amt', 'car_type']
df = pd.DataFrame(data, columns=columns)

# 2. Compute ClientStats (Replicating the SQL CTE)
# We use transform to apply the aggregation to the original rows without collapsing the dataframe
df['total_na_amt'] = df['claim_amt'].where(df['car_type'] == 'NA', 0).groupby(df['client_id']).transform('sum')
df['total_valid_amt'] = df['claim_amt'].where(df['car_type'] != 'NA', 0).groupby(df['client_id']).transform('sum')
df['valid_claim_count'] = (df['car_type'] != 'NA').astype(int).groupby(df['client_id']).transform('sum')

# 3. Apply the Allocation Logic
# To prevent division by zero warnings, we use np.where or replace 0 with NaN for the calculation
valid_amt_safe = df['total_valid_amt'].replace(0, np.nan)

conditions = [
    # Rule 1: Distribute to valid claims
    (df['car_type'] != 'NA') & (df['valid_claim_count'] > 0) & (df['total_na_amt'] > 0),
    # Rule 2: NA claim = 0 if distributed
    (df['car_type'] == 'NA') & (df['valid_claim_count'] > 0) & (df['total_na_amt'] > 0)
]

choices = [
    # Calculation for valid claims: original + proportion of NA amount
    df['claim_amt'] + ((df['claim_amt'] / valid_amt_safe) * df['total_na_amt']).fillna(0),
    # Calculation for NA claims: 0
    0
]

# Default is Rule 3: Keep original amount
df['Amt Allocation'] = np.select(conditions, choices, default=df['claim_amt'])

# 4. Round to 2 decimal places (Replicating CAST AS DECIMAL(18,2))
df['Amt Allocation'] = df['Amt Allocation'].round(2)

# 5. Cleanup and Formatting
# Drop the helper columns used for calculations
df = df.drop(columns=['total_na_amt', 'total_valid_amt', 'valid_claim_count'])

# Sort to match the image
df = df.sort_values(by=['client_id', 'Claim_id']).reset_index(drop=True)

# Optional: Format as currency strings to exactly match the screenshot
df['claim_amt_fmt'] = df['claim_amt'].apply(lambda x: f"${x:,.2f}")
df['Amt Allocation_fmt'] = df['Amt Allocation'].apply(lambda x: f"${x:,.2f}")

# Print the result
print(df[['client_id', 'Claim_id', 'car_type', 'claim_amt_fmt', 'Amt Allocation_fmt']])

    client_id   Claim_id car_type claim_amt_fmt Amt Allocation_fmt
0           1  169535646    Truck       $532.00            $549.93
1           1  734265621      Van     $5,733.00          $5,926.25
2           1  747557690      Car       $677.00            $699.82
3           1  847557635       NA       $234.00              $0.00
4           2  166535746       NA       $332.00            $332.00
5           2  235265625       NA       $733.00            $733.00
6           3  697949195      Van       $623.00          $1,310.80
7           3  797670195       NA     $2,314.00              $0.00
8           3  797969175      Van       $907.00          $1,908.33
9           3  872231801      Car       $566.00          $1,190.87
10          4  971694618    Truck       $788.00            $788.00
11          4  991694620      Van     $4,543.00          $4,543.00
12          6  215265656      Van       $888.00          $4,695.00
13          6  275330646       NA     $1,332.00              $

**Binning data (`pd.cut`):** converting continuous numbers into categorical bands.

In [ ]:
# 👉 A plain Python list of shift takings -- continuous numbers we are about to group.
#    `pd.cut` turns numbers into labelled bands. `bins` are the cut points and `labels` names them.
#   if no labels are given, the output is a categorical column with the bin ranges as labels.

## Example ##
# The marketing team doesn't want to create 100 different ads for every exact age. 
# They want to target "Teens", "Young Adults", "Middle-aged", and "Seniors".
# You use pd.cut() to turn those continuous numbers into these categorical groups.

import pandas as pd

# 1. Continuous data (exact ages)
ages = [15, 22, 34, 41, 58, 67, 19, 25, 72, 10]

# 2. Define the age cut points (bins) and the names of the groups (labels)
age_bin = [0, 18, 35, 55, 100]                                    # bins
grouping = ['Teen', 'Young Adult', 'Middle-aged', 'Senior']       # labels

# 3. Apply pd.cut()
age_groups = pd.cut(ages, bins=age_bin, labels=grouping)

print(age_groups)

# convert labels into numbers for memory saving
display(age_groups.codes)  # 0 = 1st bin/group (Teen), 1 = 2nd bin/group (Young Adult)

display(age_groups.categories)

# count how many in respective bin
pd.Series(age_groups).value_counts()

['Teen', 'Young Adult', 'Young Adult', 'Middle-aged', 'Senior', 'Senior', 'Young Adult', 'Young Adult', 'Senior', 'Teen']
Categories (4, object): ['Teen' < 'Young Adult' < 'Middle-aged' < 'Senior']


array([0, 1, 1, 2, 3, 3, 1, 1, 3, 0], dtype=int8)

Index(['Teen', 'Young Adult', 'Middle-aged', 'Senior'], dtype='object')

Young Adult    4
Senior         3
Teen           2
Middle-aged    1
dtype: int64